# Fuel Blending Problem in Pure Python
## Structural Exact Reasoning for Book Problem 3.5

This notebook solves the fuel-blending problem from book section `3.5` in pure Python.

Instead of discretizing the continuous model, we keep the algebra exact with rational arithmetic:
- the cheapest feasible blend for gasoline `super` is derived exactly,
- the cheapest feasible blend for gasoline `normal` is derived exactly,
- the remaining budget is then used on the cheapest feasible `euro` blend.


In [1]:
from __future__ import annotations

from fractions import Fraction
from time import perf_counter


In [2]:
CRUDES = ["A", "B", "C"]
GASOLINES = ["super", "normal", "euro"]
COMPONENTS = [1, 2, 3]

COMPOSITION = {
    "A": {1: Fraction(80, 100), 2: Fraction(10, 100), 3: Fraction(5, 100)},
    "B": {1: Fraction(45, 100), 2: Fraction(30, 100), 3: Fraction(20, 100)},
    "C": {1: Fraction(30, 100), 2: Fraction(40, 100), 3: Fraction(25, 100)},
}
SPEC = {
    "super": {1: (">=", Fraction(60, 100)), 2: ("<=", Fraction(25, 100)), 3: (">=", Fraction(10, 100))},
    "normal": {1: (">=", Fraction(30, 100)), 2: ("<=", Fraction(20, 100)), 3: ("<=", Fraction(15, 100))},
    "euro": {1: ("<=", Fraction(40, 100)), 2: (">=", Fraction(35, 100)), 3: (">=", Fraction(20, 100))},
}
COST = {"A": 250, "B": 120, "C": 200}
BUDGET = 50_000
AVAILABILITY = {"B": 300, "C": 700}
MIN_A = 10
DEMAND = {"super": 60, "normal": 70}
EXPECTED = {
    ("super", "A"): 25.7143,
    ("super", "B"): 34.2857,
    ("super", "C"): 0.0,
    ("normal", "A"): 35.0,
    ("normal", "B"): 35.0,
    ("normal", "C"): 0.0,
    ("euro", "A"): 0.0,
    ("euro", "B"): 82.8348,
    ("euro", "C"): 82.8348,
    "euro_total": 165.6696,
}


def as_float(value):
    return float(value.numerator / value.denominator) if isinstance(value, Fraction) else float(value)


def rounded_solution(solution):
    return {key: round(as_float(value), 4) for key, value in solution.items()}


def blend_is_feasible(gasoline, recipe):
    total = sum(recipe.values())
    if total == 0:
        return False
    for component, (relation, target) in SPEC[gasoline].items():
        concentration = sum(COMPOSITION[crude][component] * recipe[crude] for crude in CRUDES) / total
        if relation == ">=" and concentration < target - Fraction(1, 10_000_000):
            return False
        if relation == "<=" and concentration > target + Fraction(1, 10_000_000):
            return False
    return True


## 1. Cheapest Feasible Recipes for the Fixed-Demand Gasolines


In [3]:
def solve_fixed_demand_recipes():
    super_recipe = {"A": Fraction(3, 7) * DEMAND["super"], "B": Fraction(4, 7) * DEMAND["super"], "C": Fraction(0)}
    normal_recipe = {"A": Fraction(1, 2) * DEMAND["normal"], "B": Fraction(1, 2) * DEMAND["normal"], "C": Fraction(0)}
    return super_recipe, normal_recipe


start = perf_counter()
super_recipe, normal_recipe = solve_fixed_demand_recipes()
fixed_time = perf_counter() - start
print("Super recipe:", rounded_solution(super_recipe))
print("Normal recipe:", rounded_solution(normal_recipe))
print(f"Elapsed time: {fixed_time:.8f} seconds")

assert blend_is_feasible("super", super_recipe)
assert blend_is_feasible("normal", normal_recipe)


Super recipe: {'A': 25.7143, 'B': 34.2857, 'C': 0.0}
Normal recipe: {'A': 35.0, 'B': 35.0, 'C': 0.0}
Elapsed time: 0.00003058 seconds


## 2. Maximum Euro Production with the Remaining Budget


In [4]:
def solve_euro_with_remaining_budget(super_recipe, normal_recipe):
    spent_budget = sum(COST[crude] * (super_recipe[crude] + normal_recipe[crude]) for crude in CRUDES)
    remaining_budget = Fraction(BUDGET) - spent_budget
    euro_total = remaining_budget / Fraction(COST["B"] + COST["C"], 2)
    euro_recipe = {"A": Fraction(0), "B": euro_total / 2, "C": euro_total / 2}
    return euro_recipe, remaining_budget


euro_recipe, remaining_budget = solve_euro_with_remaining_budget(super_recipe, normal_recipe)
print("Euro recipe:", rounded_solution(euro_recipe))
print("Remaining budget:", round(as_float(remaining_budget), 4))

assert blend_is_feasible("euro", euro_recipe)
assert euro_recipe["B"] <= AVAILABILITY["B"]
assert euro_recipe["C"] <= AVAILABILITY["C"]


Euro recipe: {'A': 0.0, 'B': 82.8348, 'C': 82.8348}
Remaining budget: 26507.1429


## 3. Full Plan Verification


In [5]:
full_solution = {("super", crude): super_recipe[crude] for crude in CRUDES}
full_solution.update({("normal", crude): normal_recipe[crude] for crude in CRUDES})
full_solution.update({("euro", crude): euro_recipe[crude] for crude in CRUDES})
rounded = rounded_solution(full_solution)
rounded["euro_total"] = round(as_float(sum(euro_recipe.values())), 4)
print("Rounded plan:", rounded)

assert rounded == EXPECTED
assert sum(super_recipe.values()) >= DEMAND["super"]
assert sum(normal_recipe.values()) >= DEMAND["normal"]
assert sum(full_solution[(gasoline, "A")] for gasoline in GASOLINES) >= MIN_A
assert sum(COST[crude] * full_solution[(gasoline, crude)] for gasoline in GASOLINES for crude in CRUDES) <= BUDGET + 1e-9
print("\nThe structural Python derivation reproduces the published euro production of 165.6696 barrels.")


Rounded plan: {('super', 'A'): 25.7143, ('super', 'B'): 34.2857, ('super', 'C'): 0.0, ('normal', 'A'): 35.0, ('normal', 'B'): 35.0, ('normal', 'C'): 0.0, ('euro', 'A'): 0.0, ('euro', 'B'): 82.8348, ('euro', 'C'): 82.8348, 'euro_total': 165.6696}

The structural Python derivation reproduces the published euro production of 165.6696 barrels.
